# 1-2절 연습 문제 풀이

이 노트북은 1-2절 연습 문제(1-6, 1-7)의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `code_examples/ch01/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
# 환경 설정 - 시드 고정
import random

import numpy as np
import torch

SEED = 8
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 연습 문제 1-6

> [코드 1-25]에서 1일 총비용 계산의 연산 그래프를 종이에 그려 보자.

### 풀이

[코드 1-25]의 계산은 두 단계로 이루어진다.

1. `get_prices()`: `unit_costs`와 `ingredients`를 요소별로 곱한 뒤 `sum(dim=1)`로 음료별 생산 비용을 구한다.
2. `get_cost()`: 그 결과에 `sales_1day`를 곱한 뒤 `sum()`으로 1일 총비용을 구한다.

이를 [그림 1-8]과 같은 방식으로 그리면 다음과 같다.

```
  unit_costs (5,)      ingredients (2, 5)
        \                    /
         \                  /
          ( × )  요소별 곱 (브로드캐스팅으로 (2, 5))
             |
         ( ∑ dim=1 )  재료 축을 따라 집계
             |
        prices (2,)         sales_1day (2,)
             \                   /
              \                 /
               ( × )  요소별 곱 (2,)
                  |
               ( ∑ )  전체 집계
                  |
          cost_1day (스칼라)
```

그래프를 읽을 때 확인할 점은 세 가지다.

- **잎에 해당하는 텐서**는 `unit_costs`, `ingredients`, `sales_1day` 셋이다.
  이 중 자동 미분 대상으로 지정한 것은 `unit_costs` 하나뿐이므로, 역전파로 기울기가 채워지는 것도 `unit_costs.grad` 하나다.
- **화살표 방향(아래로)이 순전파**이고, 기울기는 반대 방향으로 거슬러 올라가며 계산된다.
- **마지막이 스칼라**여야 `backward()`를 호출할 수 있다. 두 번의 집계(`sum`)가 그 역할을 한다.

`unit_costs`에 대한 기울기가 `[200, 350, 100, 210, 50]`이 되는 이유도 그래프에서 읽을 수 있다.
예를 들어 우유(두 번째 요소)는 카페라테에 10씩 20잔, 코코아에 15씩 10잔 쓰이므로
`10 × 20 + 15 × 10 = 350`이 곧 우유 단위가격 1원이 총비용에 미치는 영향이다.

### 문제 검토

- **적절성: 적합.** 연산 그래프를 '읽는' 데서 '그리는' 데로 넘어가게 하는 문제다.
  본문 [그림 1-8]은 요소가 셋뿐인 단순한 그래프인데, 이 문제는 집계가 두 번 일어나고 브로드캐스팅까지 끼어 있어
  한 단계 더 나아간다. 손으로 그려 보면 '왜 마지막이 스칼라여야 하는가'가 자연스럽게 드러난다.
- **[검토] 답을 맞춰 볼 방법이 없다.** 종이에 그리는 문제라 독자가 자기 답이 맞는지 확인할 길이 없다.
  본문이 이미 `unit_costs.grad`의 값 `[200, 350, 100, 210, 50]`을 보여 주므로,
  "그린 그래프로 우유의 기울기가 350이 되는 과정을 설명해 보자"를 덧붙이면 스스로 채점할 수 있다.

**윤문안**

> **1-6**. [코드 1-25]에서 1일 총비용을 계산하는 과정의 연산 그래프를 종이에 그려 보자.
> 그리고 그린 그래프를 따라가며, 우유의 단위가격이 총비용에 미치는 영향력이 350이 되는 이유를 설명해 보자.

## 연습 문제 1-7 [도전 문제]

> 카페 주인과 재료 공급업자는 모든 재료의 단위가격을 한 번에 낮추기 어렵다고 판단했다.
> 대신 한 번 최적화할 때마다 영향력이 가장 큰 두 재료의 단위가격만 낮추기로 했다.
> 이 방식으로 20회 반복해 최적화하는 과정을 구현해 보자.
>
> 힌트: 각 반복에서 기울기를 구한 뒤 다음 반복 전에 이전 기울기를 초기화하므로,
> 그 사이에는 `grad` 속성의 값을 자유롭게 바꿔도 된다.

In [2]:
# 본문 1-2절 예제와 같은 설정
INGREDIENT_NAMES = ['커피', '우유', '초콜릿', '전기', '물']

ingredients = torch.tensor([[10, 10, 0, 8, 2], [0, 15, 10, 5, 1]])   # 음료별 재료 사용량
sales_1day = torch.tensor([20, 10])                                  # 음료별 판매량
unit_costs = torch.tensor([100., 100., 100., 100., 100.])            # 재료별 단위가격
unit_costs.requires_grad = True

def get_prices(unit_costs, ingredients):
    return (unit_costs * ingredients).sum(dim=1)

def get_cost(prices, sales):
    return (prices * sales).sum()

In [3]:
ITERATION = 20        # 최적화 반복 횟수
DISCOUNT_RATE = 0.01  # 영향력을 반영해 단위가격을 인하하는 비율
TOP_K = 2             # 한 번에 단위가격을 낮출 재료의 수

print(f'초기 비용: {get_cost(get_prices(unit_costs, ingredients), sales_1day)}')
for i in range(ITERATION):
    unit_costs.grad = None
    total_cost = get_cost(get_prices(unit_costs, ingredients), sales_1day)
    total_cost.backward()
    with torch.no_grad():
        # 영향력이 가장 큰 두 재료의 위치만 1, 나머지는 0인 마스크를 만든다
        top_indices = unit_costs.grad.topk(TOP_K).indices
        mask = torch.zeros_like(unit_costs.grad)
        mask[top_indices] = 1.
        # 마스크를 곱하면 선택되지 않은 재료의 기울기가 0이 되어 최적화에서 제외된다
        unit_costs -= DISCOUNT_RATE * unit_costs.grad * mask
        if i < 2 or i == ITERATION - 1:
            selected = ', '.join(INGREDIENT_NAMES[j] for j in top_indices.tolist())
            print(f'{i + 1}번째 최적화 - 선택된 재료: {selected}')
            print(f'    단위가격: {unit_costs.tolist()}')
            print(f'    비용: {get_cost(get_prices(unit_costs, ingredients), sales_1day)}')

초기 비용: 91000.0
1번째 최적화 - 선택된 재료: 우유, 전기
    단위가격: [100.0, 96.5, 100.0, 97.9000015258789, 100.0]
    비용: 89334.0
2번째 최적화 - 선택된 재료: 우유, 전기
    단위가격: [100.0, 93.0, 100.0, 95.80000305175781, 100.0]
    비용: 87668.0
20번째 최적화 - 선택된 재료: 우유, 전기
    단위가격: [100.0, 30.0, 100.0, 58.000030517578125, 100.0]
    비용: 57680.0078125


### 풀이 해설

핵심은 **기울기를 그대로 쓰지 않고 골라 쓰는** 것이다.
`topk()`로 기울기가 큰 두 위치를 찾아 그 자리만 1인 마스크를 만들고, 갱신식에 곱한다.
선택되지 않은 재료는 기울기가 0이 되어 단위가격이 그대로 유지된다.
힌트가 알려 주듯 `grad`는 다음 반복 시작 때 `None`으로 초기화되므로, 이렇게 값을 건드려도 뒤에 영향을 주지 않는다.

실행 결과에서 눈여겨볼 점은 **20회 내내 우유와 전기만 선택된다**는 것이다.
1일 총비용은 단위가격에 대한 일차식이라 기울기가 `[200, 350, 100, 210, 50]`으로 **항상 같다**.
단위가격이 내려가도 기울기는 변하지 않으므로 순위도 바뀌지 않는다.
본문 [코드 1-27]에서 모든 재료를 함께 낮출 때와 달리, 여기서는 세 재료의 가격이 100원에 그대로 머문다.

기울기가 파라미터 값에 따라 변하는 모델(예를 들어 1-3절의 회귀 분석 모델)이었다면
반복할 때마다 선택되는 대상이 달라졌을 것이다. 선형 모델과 비선형 모델의 차이를 확인할 수 있는 지점이다.

### 문제 검토

- **적절성: 적합. 도전 문제로서 의도가 분명하다.** `grad` 속성이 읽기 전용이 아니라 쓸 수도 있는 값이고,
  최적화 규칙을 직접 설계할 수 있다는 것을 알려 준다. 5장 이후 옵티마이저를 만나기 전에 겪어 두면 좋은 경험이다.
- **[검토] 결과가 밋밋해 보일 수 있다.** 위 해설처럼 이 예제는 기울기가 상수라 매번 같은 두 재료가 선택된다.
  독자는 '영향력이 가장 큰 두 재료를 매번 다시 고른다'는 문제 설정을 구현해 놓고도 변화를 보지 못해
  자기 구현이 틀렸다고 의심하기 쉽다. 지문이나 힌트에 한 줄을 덧붙이면 이 혼란이 사라지고,
  오히려 선형 모델의 성질을 확인하는 문제가 된다.

**윤문안**

> **1-7**. [도전 문제] 카페 주인과 재료 공급업자는 모든 재료의 단위가격을 한 번에 낮추기 어렵다고 판단했다.
> 대신 한 번 최적화할 때마다 영향력이 가장 큰 두 재료의 단위가격만 낮추기로 했다.
> 이 방식으로 20회 반복해 최적화하는 과정을 구현해 보자.
> 그리고 반복하는 동안 선택되는 재료가 어떻게 바뀌는지, 그 이유가 무엇인지도 함께 확인해 보자.

- **[검토] '영향력이 가장 큰'의 기준.** 이 예제의 기울기는 모두 양수라 문제가 없지만,
  기울기가 음수인 경우까지 생각하면 '가장 큰'이 최댓값인지 절댓값이 큰 쪽인지 모호하다.
  1-3절 예제에서는 음수 기울기가 실제로 등장하므로, 풀이에서는 절댓값 기준(`grad.abs().topk()`)도 가능하다는 점을
  해설로 덧붙일 수 있다. 지문을 고칠 필요까지는 없다고 본다.